# MANTIS — STS / Illegal Anchoring / Dark Vessels (map study)

Folium maps for studying detection SQL outcomes:

1. **STS** — active high-suspicion pairs in anchorage polygons
2. **Illegal anchoring (v2)** — stopped Class-A vessels inside restricted / watch polygons; Singapore port-limit excluded
3. **Dark vessels** — AIS silence after slow-down (`ais_vesselslowmoveactivities`)


In [34]:
import importlib

import folium
from shapely.geometry import MultiPolygon, Polygon

import sts_detection
importlib.reload(sts_detection)

from polygons import anchorage_areas
from sts_detection import (
    MAX_DISTANCE_M,
    MIN_SUSPICION_SCORE,
    detect_sts_in_anchorages,
    get_pg_engine,
)

print(f"Anchorage polygons: {len(anchorage_areas)}")
print(f"Min suspicion score: {MIN_SUSPICION_SCORE}")
print(f"Pair distance: {MAX_DISTANCE_M} m")

Anchorage polygons: 18
Min suspicion score: 4.5
Pair distance: 35.0 m


In [35]:
engine = get_pg_engine()
result = detect_sts_in_anchorages(engine)

print(f"Open high-score clusters (all areas): {result['open_high_score_count']}")
print(f"High-score clusters in anchorages:   {result['in_anchorage_cluster_count']}")
print(f"STS pairs (<= {result['max_distance_m']} m):          {result['pair_count']}")
print(f"Paired vessels (map markers only):   {result['paired_vessel_count']}")

pairs = result["pairs"]
paired_vessels = result["paired_vessels"]
payload_pairs = result["pairs_payload"]

payload_pairs

Open high-score clusters (all areas): 36
High-score clusters in anchorages:   6
STS pairs (<= 35.0 m):          6
Paired vessels (map markers only):   12


[{'observationId': 124263,
  'anchorageName': 'Batu Ampal Anchorage, Indonesia',
  'suspicionScore': 7.791,
  'distanceM': 23.195,
  'durationSeconds': 103785.053,
  'durationHours': 28.8292,
  'durationLabel': '28h 49m',
  'pairedAt': '2026-07-26 07:28:53.403215',
  'firstDetectedAt': '2026-07-25 02:39:08.350074',
  'vesselA': {'mmsi': 352006140,
   'shipName': 'PIS MENTAWAI',
   'latitude': 1.1626116666666666,
   'longitude': 103.95986,
   'sog': 0.0,
   'cog': 181.5},
  'vesselB': {'mmsi': 525108038,
   'shipName': 'PIS MENTAWAI',
   'latitude': 1.1620916666666667,
   'longitude': 103.96002666666666,
   'sog': 0.1,
   'cog': 159.7}},
 {'observationId': 127313,
  'anchorageName': 'Singapore East Anchorage, Singapore',
  'suspicionScore': 5.249,
  'distanceM': 12.977,
  'durationSeconds': 10631.216,
  'durationHours': 2.9531,
  'durationLabel': '2h 57m',
  'pairedAt': '2026-07-26 07:28:53.403215',
  'firstDetectedAt': '2026-07-26 04:31:42.187211',
  'vesselA': {'mmsi': 566841000,
   '

## Map — anchorage polygons + paired vessels only

In [36]:
colors = [
    "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd",
    "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf",
    "#393b79", "#637939", "#8c6d31", "#843c39", "#7b4173",
    "#3182bd", "#e6550d", "#31a354",
]

shapes = [Polygon([(lon, lat) for lon, lat in area["polygon"]]) for area in anchorage_areas]
bounds = MultiPolygon(shapes).bounds
center_lat = (bounds[1] + bounds[3]) / 2
center_lon = (bounds[0] + bounds[2]) / 2

m_sts = folium.Map(location=[center_lat, center_lon], zoom_start=9, tiles="OpenStreetMap")

for i, area in enumerate(anchorage_areas):
    color = colors[i % len(colors)]
    coords_latlon = [[lat, lon] for lon, lat in area["polygon"]]
    folium.Polygon(
        locations=coords_latlon,
        color=color,
        weight=1,
        fill=True,
        fill_color=color,
        fill_opacity=0.15,
        tooltip=area["name"],
    ).add_to(m_sts)

for _, v in paired_vessels.iterrows():
    sog = v.get("sog")
    cog = v.get("cog")
    folium.CircleMarker(
        location=[float(v["curlatitude"]), float(v["curlongitude"])],
        radius=7,
        color="#d62728",
        fill=True,
        fill_color="#d62728",
        fill_opacity=0.9,
        tooltip=(
            f"{v.get('shipname', '')} ({v['mmsi']})\n"
            f"{v.get('anchorage_name', '')}\n"
            f"sog={sog} cog={cog}\n"
            f"score={v.get('suspicion_score', '')}"
        ),
    ).add_to(m_sts)

for p in payload_pairs:
    a, b = p["vesselA"], p["vesselB"]
    folium.PolyLine(
        locations=[
            [a["latitude"], a["longitude"]],
            [b["latitude"], b["longitude"]],
        ],
        color="#d62728",
        weight=4,
        opacity=0.95,
        tooltip=(
            f"{p['anchorageName']}: {a['shipName']} ↔ {b['shipName']}\n"
            f"{p['distanceM']:.1f} m | score={p['suspicionScore']}\n"
            f"duration={p['durationLabel']} | pairedAt={p['pairedAt']}"
        ),
    ).add_to(m_sts)

if not paired_vessels.empty:
    m_sts.fit_bounds([
        [float(paired_vessels["curlatitude"].min()), float(paired_vessels["curlongitude"].min())],
        [float(paired_vessels["curlatitude"].max()), float(paired_vessels["curlongitude"].max())],
    ])
else:
    m_sts.fit_bounds([[bounds[1], bounds[0]], [bounds[3], bounds[2]]])

m_sts

---

## Illegal anchoring (v2)

Stopped / stale **shipType 70–89** vessels **inside** restricted-limit **or** watch polygons from `polygons.py`.  
**Excluded:** Singapore port-limit polygons (East / Western OPL / South + Excl*).


In [37]:
import illegal_anchoring
importlib.reload(illegal_anchoring)

from illegal_anchoring import (
    RESTRICTED_LIMIT_LONLAT,
    detect_illegal_anchoring,
    port_limit_polygons,
    watch_polygons,
)

engine = get_pg_engine()
illegal_result = detect_illegal_anchoring(engine)
illegal_payload = illegal_result["vessels_payload"]
watch_areas = watch_polygons()
port_areas = port_limit_polygons()

print(f"Stopped candidates: {illegal_result['stopped_candidate_count']}")
print(f"Illegal / suspect:  {illegal_result['illegal_count']}")
print(f"By reason:          {illegal_result['by_reason']}")
print(f"Watch polygons:     {illegal_result['watch_polygon_count']}")
print(f"Port-limit polys:   {illegal_result['port_limit_polygon_count']}")
print(f"Rule:               {illegal_result['rule_version']}")
print("Port-limit names:", [a["name"] for a in port_areas])

illegal_payload[:5] if illegal_payload else []


Stopped candidates: 1720
Illegal / suspect:  297
By reason:          {'in_restricted_zone+in_watch_polygon': 166, 'in_watch_polygon': 119, 'in_restricted_zone': 12}
Watch polygons:     10
Port-limit polys:   8
Rule:               v2-in-restricted-or-watch-exclude-port-limit
Port-limit names: ['Singapore East Anchorage, Singapore', 'Singapore Western OPL, Singapore', 'Singapore South Anchorage, Singapore', 'Singapore South Anchorage (Excl1), Singapore', 'Singapore South Anchorage (Excl2), Singapore', 'Singapore South Anchorage (Excl3), Singapore', 'Singapore South Anchorage (Excl4), Singapore', 'Singapore South Anchorage (Excl5), Singapore']


[{'mmsi': 205658000,
  'shipName': 'CYPRES',
  'shipType': 80,
  'shipTypeDesc': 'Tanker all ships of this type',
  'latitude': 1.3068716666666667,
  'longitude': 104.18020166666666,
  'sog': 0.0,
  'cog': 201.4,
  'navStatusDesc': 'At anchor',
  'tsStop': '2026-07-15 22:42:26.904000',
  'tsCurrent': '2026-07-26 07:27:26.949000',
  'durationSeconds': 895500.045,
  'durationHours': 248.75,
  'durationLabel': '248h 45m',
  'inRestrictedZone': False,
  'inWatchPolygon': True,
  'inPortLimit': False,
  'watchPolygonName': 'Pasir Gudang, Johor Anchorage Malaysia',
  'portLimitName': None,
  'reason': 'in_watch_polygon',
  'detectedAt': '2026-07-26T07:29:19.350213+00:00'},
 {'mmsi': 257972000,
  'shipName': 'BERLINDA',
  'shipType': 70,
  'shipTypeDesc': 'Cargo all ships of this type',
  'latitude': 1.328475,
  'longitude': 104.27635166666667,
  'sog': 0.1,
  'cog': 204.9,
  'navStatusDesc': 'At anchor',
  'tsStop': '2026-07-26 06:22:21.523000',
  'tsCurrent': '2026-07-26 07:22:21.838000',
 

### Map — watch polygons + restricted + port limit (excluded) + candidates

Layers:
- **blue** fill — watch polygons (kept if vessel inside)
- **green** outline — Singapore port limit (excluded)
- **red** outline — restricted-limit zone

Markers:
- **orange** — `in_restricted_zone`
- **red** — `in_watch_polygon`
- **purple** — both


In [38]:
REASON_COLORS = {
    "in_restricted_zone": "#ff7f0e",
    "in_watch_polygon": "#d62728",
    "in_restricted_zone+in_watch_polygon": "#9467bd",
    "in_watch_polygon+in_restricted_zone": "#9467bd",
}

poly_shapes = [Polygon([(lon, lat) for lon, lat in area["polygon"]]) for area in anchorage_areas]
poly_bounds = MultiPolygon(poly_shapes).bounds
map_center = [(poly_bounds[1] + poly_bounds[3]) / 2, (poly_bounds[0] + poly_bounds[2]) / 2]

m_illegal = folium.Map(location=map_center, zoom_start=9, tiles="OpenStreetMap")

for area in watch_areas:
    coords_latlon = [[lat, lon] for lon, lat in area["polygon"]]
    folium.Polygon(
        locations=coords_latlon,
        color="#3182bd",
        weight=1,
        fill=True,
        fill_color="#3182bd",
        fill_opacity=0.15,
        tooltip=f"Watch: {area['name']}",
    ).add_to(m_illegal)

for area in port_areas:
    coords_latlon = [[lat, lon] for lon, lat in area["polygon"]]
    folium.Polygon(
        locations=coords_latlon,
        color="#2ca02c",
        weight=2,
        fill=True,
        fill_color="#2ca02c",
        fill_opacity=0.08,
        tooltip=f"Port limit (excluded): {area['name']}",
    ).add_to(m_illegal)

restricted_latlon = [[lat, lon] for lon, lat in RESTRICTED_LIMIT_LONLAT]
folium.Polygon(
    locations=restricted_latlon,
    color="#e31a1c",
    weight=2,
    fill=True,
    fill_color="#e31a1c",
    fill_opacity=0.08,
    tooltip="Restricted limit zone",
).add_to(m_illegal)

for v in illegal_payload:
    reason = v.get("reason") or "unknown"
    color = REASON_COLORS.get(reason, "#333333")
    folium.CircleMarker(
        location=[v["latitude"], v["longitude"]],
        radius=6,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.85,
        tooltip=(
            f"{v.get('shipName') or ''} ({v['mmsi']})\n"
            f"{v.get('shipTypeDesc') or ''}\n"
            f"reason={reason}\n"
            f"watch={v.get('watchPolygonName')}\n"
            f"sog={v.get('sog')} cog={v.get('cog')}\n"
            f"duration={v.get('durationLabel')}\n"
            f"inRestricted={v.get('inRestrictedZone')} inWatch={v.get('inWatchPolygon')}"
        ),
    ).add_to(m_illegal)

if illegal_payload:
    lats = [v["latitude"] for v in illegal_payload]
    lons = [v["longitude"] for v in illegal_payload]
    m_illegal.fit_bounds([[min(lats), min(lons)], [max(lats), max(lons)]])
else:
    m_illegal.fit_bounds([[poly_bounds[1], poly_bounds[0]], [poly_bounds[3], poly_bounds[2]]])

m_illegal


---

## Dark / AIS-transponder-off vessels (v1)

Open slow-move activities where the vessel went silent **before** a confirmed stop (`rowcount < 30`), silence ≥ 30 minutes, shipType **70–89**. Independent of anchorage polygons (shown only as context).

Set `INCLUDE_COVERAGE_EXIT = False` for the ops-tight list.

In [39]:
import dark_vessels
importlib.reload(dark_vessels)

from dark_vessels import detect_dark_vessels

# True = include possible_coverage_exit (research); False = ops-tight
INCLUDE_COVERAGE_EXIT = True

engine = get_pg_engine()
dark_result = detect_dark_vessels(engine, include_coverage_exit=INCLUDE_COVERAGE_EXIT)
dark_payload = dark_result["vessels_payload"]

print(f"Candidates:            {dark_result['candidate_count']}")
print(f"By reason:             {dark_result['by_reason']}")
print(f"By confidence:         {dark_result['by_confidence']}")
print(f"includeCoverageExit:   {dark_result['include_coverage_exit']}")
print(f"minSilenceMinutes:     {dark_result['min_silence_minutes']}")
print(f"Rule:                  {dark_result['rule_version']}")

dark_payload[:5] if dark_payload else []


Candidates:            448
By reason:             {'possible_coverage_exit': 289, 'low_evidence_ais_gap': 82, 'suspected_dark_after_slowdown': 77}
By confidence:         {'low': 289, 'medium': 82, 'high': 77}
includeCoverageExit:   True
minSilenceMinutes:     30
Rule:                  v1-slowmove-dark-after-slowdown


[{'activityId': 17,
  'mmsi': 671185100,
  'shipName': 'NATTO 1',
  'shipType': 70,
  'shipTypeDesc': 'Cargo all ships of this type',
  'latitude': 4.006495,
  'longitude': 101.00982666666667,
  'sog': 0.0,
  'cog': 128.2,
  'navStatusDesc': 'Under way using engine',
  'rowCount': 1,
  'distanceM': 0.0,
  'ts': '2026-05-20 23:39:55.308000',
  'tsStop': '2026-05-20 23:39:55.308000',
  'tsCurrent': '2026-05-20 23:39:55.308000',
  'silenceSeconds': 5730581.686,
  'silenceHours': 1591.8282,
  'silenceLabel': '1591h 49m',
  'darkReason': 'possible_coverage_exit',
  'confidence': 'low'},
 {'activityId': 100,
  'mmsi': 538009769,
  'shipName': 'AL WAJBAH',
  'shipType': 80,
  'shipTypeDesc': 'Tanker all ships of this type',
  'latitude': 5.15186,
  'longitude': 115.47672666666666,
  'sog': 0.0,
  'cog': 160.8,
  'navStatusDesc': 'At anchor',
  'rowCount': 1,
  'distanceM': 0.0,
  'ts': '2026-05-22 00:30:37.903000',
  'tsStop': '2026-05-22 00:30:37.903000',
  'tsCurrent': '2026-05-22 00:30:37.

### Map — last known position by `darkReason`

- **red** — `suspected_dark_after_slowdown` (higher interest)
- **gray** — `possible_coverage_exit` (SEA footprint leave / competing explanation)
- **orange** — `low_evidence_ais_gap`


In [40]:
DARK_COLORS = {
    "suspected_dark_after_slowdown": "#d62728",
    "possible_coverage_exit": "#7f7f7f",
    "low_evidence_ais_gap": "#ff7f0e",
}

map_colors = [
    "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd",
    "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf",
    "#393b79", "#637939", "#8c6d31", "#843c39", "#7b4173",
    "#3182bd", "#e6550d", "#31a354",
]
poly_shapes = [Polygon([(lon, lat) for lon, lat in area["polygon"]]) for area in anchorage_areas]
poly_bounds = MultiPolygon(poly_shapes).bounds
map_center = [(poly_bounds[1] + poly_bounds[3]) / 2, (poly_bounds[0] + poly_bounds[2]) / 2]

m_dark = folium.Map(location=map_center, zoom_start=8, tiles="OpenStreetMap")

# Anchorage polygons as light context only (dark detection does not use them)
for i, area in enumerate(anchorage_areas):
    color = map_colors[i % len(map_colors)]
    coords_latlon = [[lat, lon] for lon, lat in area["polygon"]]
    folium.Polygon(
        locations=coords_latlon,
        color=color,
        weight=1,
        fill=True,
        fill_color=color,
        fill_opacity=0.08,
        tooltip=area["name"],
    ).add_to(m_dark)

plotted = 0
for v in dark_payload:
    lat, lon = v.get("latitude"), v.get("longitude")
    if lat is None or lon is None:
        continue
    reason = v.get("darkReason") or "unknown"
    color = DARK_COLORS.get(reason, "#333333")
    folium.CircleMarker(
        location=[lat, lon],
        radius=5,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.8,
        tooltip=(
            f"{v.get('shipName') or ''} ({v['mmsi']})\n"
            f"{v.get('shipTypeDesc') or ''}\n"
            f"reason={reason} | confidence={v.get('confidence')}\n"
            f"silence={v.get('silenceLabel')} | rowCount={v.get('rowCount')}\n"
            f"sog={v.get('sog')} cog={v.get('cog')}\n"
            f"tsCurrent={v.get('tsCurrent')}"
        ),
    ).add_to(m_dark)
    plotted += 1

print(f"Plotted markers: {plotted} / {len(dark_payload)}")

if plotted:
    lats = [v["latitude"] for v in dark_payload if v.get("latitude") is not None]
    lons = [v["longitude"] for v in dark_payload if v.get("longitude") is not None]
    m_dark.fit_bounds([[min(lats), min(lons)], [max(lats), max(lons)]])
else:
    m_dark.fit_bounds([[poly_bounds[1], poly_bounds[0]], [poly_bounds[3], poly_bounds[2]]])

m_dark


Plotted markers: 448 / 448
